In [62]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DirectoryLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

import numpy as np
from typing import List

c:\Users\Sam Ben-Yosef\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data_dir = "../data_dir"
loader = DirectoryLoader(
    data_dir, 
    glob="**/*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
documents = loader.load()
print(f"Loaded {len(documents)} documents from {data_dir}")

Loaded 4 documents from ../data_dir


In [4]:
# Document splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print("Sample chunk:")
print(chunks[0]) 
print("\n" + "="*50 + "\n")

Split into 101 chunks
Sample chunk:
page_content='COOKIE RECIPE COLLECTION
All recipes yield approximately 24–36 cookies unless noted. Oven temperatures are for conventional ovens.' metadata={'source': '..\\data_dir\\cookie_recipes.txt'}




In [5]:
# Embedding generation
sample_text = "What food is good with coffee?"
embeddings = OpenAIEmbeddings()
embeddings 

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000258852C6660>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000258852C6F90>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [6]:
chunks

[Document(metadata={'source': '..\\data_dir\\cookie_recipes.txt'}, page_content='COOKIE RECIPE COLLECTION\n========================\nAll recipes yield approximately 24–36 cookies unless noted. Oven temperatures are for conventional ovens.'),
 Document(metadata={'source': '..\\data_dir\\cookie_recipes.txt'}, page_content='------------------------------------------------------------\n1) CLASSIC CHOCOLATE CHIP COOKIES\n------------------------------------------------------------\nIngredients\n- 1 cup (226 g) unsalted butter, softened\n- 3/4 cup (150 g) granulated sugar\n- 3/4 cup (165 g) packed brown sugar\n- 2 large eggs\n- 2 tsp vanilla extract\n- 2 1/4 cups (280 g) all-purpose flour\n- 1 tsp baking soda\n- 1 tsp fine salt\n- 2 cups (340 g) semisweet chocolate chips'),
 Document(metadata={'source': '..\\data_dir\\cookie_recipes.txt'}, page_content='Instructions\n1. Heat oven to 350°F (175°C). Line 2 baking sheets with parchment.\n2. Cream butter and both sugars until fluffy, 2–3 minutes

In [7]:
vector = embeddings.embed_query(sample_text)
print(f"Generated embedding of length {len(vector)}")
print("Sample embedding values:")   
print(vector[:10])  # Print first 10 values of the embedding
print("\n" + "="*50 + "\n")

Generated embedding of length 1536
Sample embedding values:
[0.009406364522874355, -0.026151204481720924, 0.0286225825548172, -0.00846068374812603, -0.010812275111675262, -0.007741967216134071, -0.004451000597327948, -0.02592424303293228, -0.006206813268363476, -0.012249709106981754]




In [8]:
# Initialize the chromadb vector store and store the chunks in vector representations
persistent_directory = "../chroma_db"
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persistent_directory,
    collection_name="documents_collection"
)

print(f"Vector store created with collection name 'documents_collection' at {persistent_directory}")

Vector store created with collection name 'documents_collection' at ../chroma_db


In [9]:
# Test the similarity search
query = "What is the best cookie to have with coffee with cream and sugar?"
similar_docs = vector_store.similarity_search(query, k=3)
print(f"Top 3 similar documents for the query: '{query}'")
for i, doc in enumerate(similar_docs):
    print(f"\nDocument {i+1}:\n{doc.page_content}\n")

Top 3 similar documents for the query: 'What is the best cookie to have with coffee with cream and sugar?'

Document 1:
------------------------------------------------------------
21) OATMEAL RAISIN COOKIES (CAFÉ CLASSIC)
------------------------------------------------------------
Ingredients
- 1 cup butter, softened
- 1 cup brown sugar + 1/2 cup sugar
- 2 eggs + 2 tsp vanilla
- 1 1/2 cups flour
- 1 tsp baking soda + 1 tsp cinnamon + 1/2 tsp salt
- 3 cups oats + 1 cup raisins
Instructions
1. Bake 350°F (175°C) 10–12 min.


Document 2:
------------------------------------------------------------
13) ESPRESSO CHOCOLATE CHIP COOKIES
------------------------------------------------------------
Ingredients
- 1 cup (226 g) unsalted butter, softened
- 3/4 cup (150 g) sugar
- 3/4 cup (165 g) brown sugar
- 2 large eggs
- 2 tsp vanilla
- 2 1/4 cups (280 g) flour
- 1 tsp baking soda
- 1 tsp fine salt
- 1–2 Tbsp instant espresso powder (to taste)
- 2 cups (340 g) chocolate chips


Document 3:
--

In [10]:
# Test the similarity search
query = "What is the recipe for a good coffee pastry?"
similar_docs = vector_store.similarity_search(query, k=3)
print(f"Top 3 similar documents for the query: '{query}'")
for i, doc in enumerate(similar_docs):
    print(f"\nDocument {i+1}:\n{doc.page_content}\n")

Top 3 similar documents for the query: 'What is the recipe for a good coffee pastry?'

Document 1:
CAFÉ PASTRIES RECIPE COLLECTION
Notes
- Butter temperature matters: “cold” for flaky pastries; “room temp” for cakes/cookies.
- For laminated dough (croissants/danish), keep dough and butter cool at all times.
- Oven temps are for conventional ovens.


Document 2:
------------------------------------------------------------
2) FRENCH PRESS COFFEE
------------------------------------------------------------
Ingredients
- 30 g coarse-ground coffee
- 500 g water (200°F / 93°C)
Instructions
1. Add coffee and water. Stir.
2. Steep 4 minutes. Break crust and skim foam.
3. Press slowly. Serve immediately.


Document 3:
------------------------------------------------------------
13) DANISH PASTRY CHEESE POCKETS (QUICK PUFF)
------------------------------------------------------------
Ingredients
- 1 sheet puff pastry (thawed but cold)
Filling
- 8 oz (225 g) cream cheese
- 1/4 cup (50 g) sugar
- 

In [11]:
# Test the similarity search
query = "What does RAG stand for?"
similar_docs = vector_store.similarity_search(query, k=3)
print(f"Top 3 similar documents for the query: '{query}'")
for i, doc in enumerate(similar_docs):
    print(f"\nDocument {i+1}:\n{doc.page_content}\n")

Top 3 similar documents for the query: 'What does RAG stand for?'

Document 1:
Evaluation and tuning for RAG are best framed as an information retrieval problem followed by a generation problem. Retrieval metrics include recall@k (does the gold passage appear in top-k) and nDCG (rank quality). Generation metrics may include correctness or exact-match scoring for structured questions, but engineers often rely on task-specific checks (e.g., whether a generated answer references the correct identifier). A highly actionable approach is to build a labeled dataset of queries


Document 2:
Evaluation and tuning for RAG are best framed as an information retrieval problem followed by a generation problem. Retrieval metrics include recall@k (does the gold passage appear in top-k) and nDCG (rank quality). Generation metrics may include correctness or exact-match scoring for structured questions, but engineers often rely on task-specific checks (e.g., whether a generated answer references the corr

In [12]:
# Test the similarity search
query = "What is a transformer?"
similar_docs = vector_store.similarity_search(query, k=3)
print(f"Top 3 similar documents for the query: '{query}'")
for i, doc in enumerate(similar_docs):
    print(f"\nDocument {i+1}:\n{doc.page_content}\n")

Top 3 similar documents for the query: 'What is a transformer?'

Document 1:
Most modern text-based generative systems are built on transformer architectures. A transformer processes sequences using self-attention, a mechanism that lets the model weigh relationships between tokens (subword units) regardless of distance in the input. This enables the model to capture long-range dependencies, such as how an early clause constrains later grammar or how a requirement in a spec should influence a later implementation detail. When teaching transformers, it is useful to


Document 2:
Most modern text-based generative systems are built on transformer architectures. A transformer processes sequences using self-attention, a mechanism that lets the model weigh relationships between tokens (subword units) regardless of distance in the input. This enables the model to capture long-range dependencies, such as how an early clause constrains later grammar or how a requirement in a spec should influenc

In [13]:
# Advanced similarity search with scores
query = "Explain the concept of retrieval-augmented generation."
results_with_scores = vector_store.similarity_search_with_score(query, k=3)
print(f"Top 3 similar documents with scores for the query: '{query}'")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\nDocument {i+1} (Score: {score}):\n{doc.page_content}\n")

Top 3 similar documents with scores for the query: 'Explain the concept of retrieval-augmented generation.'

Document 1 (Score: 0.18068479001522064):
Retrieval-augmented generation is an engineering pattern that couples a generative model with an external knowledge store, typically to improve factual grounding, reduce prompt length by referencing only relevant passages, and enable updates without retraining the base model. The system can be treated as a pipeline with distinct stages: document ingestion, chunking, embedding, indexing, retrieval, context assembly, and generation. Each stage has quality and performance knobs, and production


Document 2 (Score: 0.18071895837783813):
Retrieval-augmented generation is an engineering pattern that couples a generative model with an external knowledge store, typically to improve factual grounding, reduce prompt length by referencing only relevant passages, and enable updates without retraining the base model. The system can be treated as a pip

## Note on similarity scores, L2 and cosine
In ChromaDB, the similarity scores returned by the `similarity_search_with_score` method are based on the L2 distance between vectors. A lower L2 distance indicates a higher similarity between the query and the document vectors.

Cosine similarity, on the other hand, measures the cosine of the angle between two vectors. It ranges from -1 to 1, where 1 means the vectors are identical in direction, 0 means they are orthogonal (no similarity), and -1 means they are diametrically opposed.

When using L2 distance, a smaller score indicates greater similarity, whereas with cosine similarity, a larger score indicates greater similarity. It's important to keep this distinction in mind when interpreting the results of similarity searches.

In [14]:
# Initialize the LLM with retrieval capability
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.2,
    model_name="gpt-3.5-turbo",
    max_completion_tokens=500
)

In [15]:
test_response = llm.invoke("What is a large language model?")
print(test_response)

content="A large language model is a type of artificial intelligence system that is trained on vast amounts of text data in order to understand and generate human language. These models are capable of performing a wide range of natural language processing tasks, such as text generation, translation, summarization, and more. Large language models are typically built using deep learning techniques, such as neural networks, and require significant computational resources to train and run. Examples of large language models include OpenAI's GPT-3 and Google's BERT." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 14, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': N

In [16]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-3.5-turbo")
llm

ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000025886565E50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000025886566210>, root_client=<openai.OpenAI object at 0x0000025886565BD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000025886565F90>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [17]:
llm.invoke("What is a large language model?")

AIMessage(content='A large language model is a type of artificial intelligence that has been trained on a vast amount of text data in order to understand and generate human language. These models are typically neural networks that can process and generate text at a large scale, with the ability to understand context, grammar, and syntax. Large language models have been used in a variety of applications, including natural language processing tasks such as machine translation, chatbots, and text generation. The most famous examples of large language models include GPT-3 (Generative Pre-trained Transformer 3) developed by OpenAI and BERT (Bidirectional Encoder Representations from Transformers) developed by Google.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 14, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0

In [18]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [19]:
# Convert vector store to retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [20]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000258852C7620>, search_kwargs={'k': 3})

In [21]:
from langchain_core.prompts import ChatPromptTemplate
# Create a custom prompt template
system_prompt = "You are an expert assistant. Use the following context to answer the question at the end.\n\nContext: {context}"

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{input}")
])

In [22]:
prompt 

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an expert assistant. Use the following context to answer the question at the end.\n\nContext: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [23]:
# Create the document chain that combines retrieved docs with the LLM
document_chain = create_stuff_documents_chain(llm, prompt)

# Create the retrieval chain
rag_chain = create_retrieval_chain(retriever, document_chain)

In [24]:
response = rag_chain.invoke({"input": "What is GenAI and how is it used? What technologies does it leverage?"})
print(response["answer"])

GenAI is a system developed by GenAI Engineering that leverages advanced technologies such as transformers, tokenization, and inference mechanics. Transformers are models that process sequential data by considering the context of each word in a sentence. Tokenization is the process of converting text into smaller units called tokens for analysis. Inference mechanics refer to the methods used to generate predictions or conclusions based on data inputs. GenAI uses these technologies to perform various tasks such as natural language processing, image recognition, and data analysis in a wide range of applications including chatbots, recommendation systems, and automated decision-making processes.


In [25]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an expert assistant. Use the following context to answer the question at the end.\n\nContext: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': 

In [26]:
# Create final RAG chain
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000258852C7620>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an expert assistant. Use the following context to answer the question at the end.\n\nContext: {context}'), additional_kwar

In [27]:
response = rag_chain.invoke({"input": "What is generative AI and how is it used? What technologies does it leverage?"})

In [28]:
response

{'input': 'What is generative AI and how is it used? What technologies does it leverage?',
 'context': [Document(metadata={'source': '..\\data_dir\\genai_foundations.txt'}, page_content='Generative AI refers to a family of machine learning systems that learn statistical patterns from data and then produce new content—text, images, audio, code, or structured outputs—that resembles what they learned. In teaching the topic, it helps to start with the core idea that these models do not “retrieve” answers in the same way a search engine does. Instead, they estimate the most likely continuation (or completion) of an input based on the distribution they learned during training, and'),
  Document(metadata={'source': '..\\data_dir\\genai_foundations.txt'}, page_content='Generative AI refers to a family of machine learning systems that learn statistical patterns from data and then produce new content—text, images, audio, code, or structured outputs—that resembles what they learned. In teaching t

In [29]:
response["answer"]

'Generative AI refers to machine learning systems that learn statistical patterns from data and then generate new content such as text, images, audio, code, or structured outputs that resemble what they learned. These systems do not "retrieve" answers like search engines but estimate the most likely continuation of an input based on their training data distribution.\n\nGenerative AI can be used in various applications such as creating realistic images, generating human-like text, composing music, or even designing products. Some technologies leveraged by generative AI include deep learning algorithms like Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), and Transformers. These technologies enable the models to learn complex patterns and generate diverse and high-quality outputs.'

In [30]:
# If you need output parsers, use these instead of the deprecated StructuredOutputParser
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough

In [31]:
create_prompt = ChatPromptTemplate.from_template(
    """Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know, don't try to make up an answer. Provide specific details.
Context: {context}
Question: {question}
Answer:""")

In [32]:
# Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [33]:
from langchain_core.output_parsers import StrOutputParser

# Build the chain using LCEL
rag_chain_lcel = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | create_prompt 
    | llm 
    | StrOutputParser()
)

response = rag_chain_lcel.invoke("What is generative AI and how is it used? What technologies does it leverage?")

In [34]:
response

'Generative AI is a type of machine learning system that learns statistical patterns from data and uses that learning to produce new content such as text, images, audio, code, or structured outputs that resemble what they have learned. It does not retrieve answers like a search engine, but rather estimates the most likely continuation of an input based on the distribution learned during training. Generative AI leverages technologies such as neural networks, deep learning, natural language processing, and computer vision.'

In [35]:
retriever.invoke("What is generative AI and how is it used? What technologies does it leverage?")

[Document(metadata={'source': '..\\data_dir\\genai_foundations.txt'}, page_content='Generative AI refers to a family of machine learning systems that learn statistical patterns from data and then produce new content—text, images, audio, code, or structured outputs—that resembles what they learned. In teaching the topic, it helps to start with the core idea that these models do not “retrieve” answers in the same way a search engine does. Instead, they estimate the most likely continuation (or completion) of an input based on the distribution they learned during training, and'),
 Document(metadata={'source': '..\\data_dir\\genai_foundations.txt'}, page_content='Generative AI refers to a family of machine learning systems that learn statistical patterns from data and then produce new content—text, images, audio, code, or structured outputs—that resembles what they learned. In teaching the topic, it helps to start with the core idea that these models do not “retrieve” answers in the same w

In [36]:
new_document = """
AI companions are applications that leverage artificial intelligence technologies to provide assistance, support, or companionship to users. These AI companions can take various forms, including chatbots, virtual assistants, and social robots. They are designed to interact with users in a human-like manner
, often using natural language processing and machine learning techniques.
Some common features and functionalities of AI companions include:
1. Natural Language Processing (NLP): AI companions can understand and respond to human language, allowing for seamless communication through text or voice interactions.
2. Personalization: AI companions can learn from user interactions and preferences to provide personalized experiences, recommendations, and responses.
3. Emotional Intelligence: Some AI companions are designed to recognize and respond to human emotions, providing empathetic support and companionship.
4. Task Assistance: AI companions can help users with various tasks, such as setting reminders, managing schedules, providing information, and controlling smart home devices.
5. Entertainment: Many AI companions offer entertainment features, such as playing games, telling jokes, or engaging in casual conversations.

AI companions are increasingly being used in various domains, including healthcare, education, customer service, and personal well-being. They have the potential to enhance user experiences and provide valuable support in daily life.
There are concerns regarding privacy, security, and ethical considerations when it comes to AI companions, as they often collect and process personal data. It is essential to ensure that these applications are designed and used responsibly to protect user rights and well-being.
Replika is an example of an AI companion that uses advanced AI techniques to provide users with a personalized and interactive experience. It is designed to be a supportive and empathetic companion that can engage in meaningful conversations and help users with their emotional well-being.
Kindroid is another example of an AI companion that focuses on providing companionship and support to users. It leverages AI technologies to create a virtual friend that can interact with users in a human-like manner
and provide emotional support.

Ethical considerations in the development and use of AI companions include ensuring user privacy, preventing misuse, and addressing potential biases in AI algorithms. It is crucial to establish guidelines and regulations to govern the responsible use of AI companions in society.
People can become attached to AI companions, forming emotional bonds with these virtual entities. This attachment can provide comfort and support, especially for individuals who may feel isolated or lonely. However, it is essential to maintain a balance between virtual companionship and real-world social interactions to ensure overall well-being.
"""

In [37]:
chunks

[Document(metadata={'source': '..\\data_dir\\cookie_recipes.txt'}, page_content='COOKIE RECIPE COLLECTION\n========================\nAll recipes yield approximately 24–36 cookies unless noted. Oven temperatures are for conventional ovens.'),
 Document(metadata={'source': '..\\data_dir\\cookie_recipes.txt'}, page_content='------------------------------------------------------------\n1) CLASSIC CHOCOLATE CHIP COOKIES\n------------------------------------------------------------\nIngredients\n- 1 cup (226 g) unsalted butter, softened\n- 3/4 cup (150 g) granulated sugar\n- 3/4 cup (165 g) packed brown sugar\n- 2 large eggs\n- 2 tsp vanilla extract\n- 2 1/4 cups (280 g) all-purpose flour\n- 1 tsp baking soda\n- 1 tsp fine salt\n- 2 cups (340 g) semisweet chocolate chips'),
 Document(metadata={'source': '..\\data_dir\\cookie_recipes.txt'}, page_content='Instructions\n1. Heat oven to 350°F (175°C). Line 2 baking sheets with parchment.\n2. Cream butter and both sugars until fluffy, 2–3 minutes

In [38]:
new_doc = Document(
    page_content=new_document,
    metadata={"source": "generated_ai_companion_info.txt", "topic": "AI Companions"}
    )

In [39]:
new_doc

Document(metadata={'source': 'generated_ai_companion_info.txt', 'topic': 'AI Companions'}, page_content='\nAI companions are applications that leverage artificial intelligence technologies to provide assistance, support, or companionship to users. These AI companions can take various forms, including chatbots, virtual assistants, and social robots. They are designed to interact with users in a human-like manner\n, often using natural language processing and machine learning techniques.\nSome common features and functionalities of AI companions include:\n1. Natural Language Processing (NLP): AI companions can understand and respond to human language, allowing for seamless communication through text or voice interactions.\n2. Personalization: AI companions can learn from user interactions and preferences to provide personalized experiences, recommendations, and responses.\n3. Emotional Intelligence: Some AI companions are designed to recognize and respond to human emotions, providing emp

In [40]:
# Add new document to the vector store
new_chunks = text_splitter.split_documents([new_doc])
print(f"Adding {len(new_chunks)} new chunks to the vector store.")

Adding 9 new chunks to the vector store.


In [41]:
new_chunks

[Document(metadata={'source': 'generated_ai_companion_info.txt', 'topic': 'AI Companions'}, page_content='AI companions are applications that leverage artificial intelligence technologies to provide assistance, support, or companionship to users. These AI companions can take various forms, including chatbots, virtual assistants, and social robots. They are designed to interact with users in a human-like manner\n, often using natural language processing and machine learning techniques.\nSome common features and functionalities of AI companions include:'),
 Document(metadata={'source': 'generated_ai_companion_info.txt', 'topic': 'AI Companions'}, page_content='1. Natural Language Processing (NLP): AI companions can understand and respond to human language, allowing for seamless communication through text or voice interactions.\n2. Personalization: AI companions can learn from user interactions and preferences to provide personalized experiences, recommendations, and responses.\n3. Emotio

In [42]:
vector_store.add_documents(new_chunks) 

['145a1866-182b-4a62-9971-427dda12585b',
 '7dd669eb-1ccb-4ace-9fde-a1a228bf99ec',
 '49c74ab6-7ad0-4978-b902-6160b49bcfd0',
 'e1529c94-f3c6-4a09-9cbb-84239eadbaaf',
 'a2c226b4-3e0e-43dd-979b-649de55ec95b',
 '2304dc7c-72d4-40c3-8571-9702c91db7ad',
 '1fc56613-2733-4d92-a6a8-9f4471bb67f2',
 'c7a7c56b-4a7d-4433-9544-84dc7567e428',
 'ef25e6bc-c247-4426-87cf-0c2aa7c35a23']

In [43]:
new_question = "What are AI companions and how do they work?"
new_result = rag_chain_lcel.invoke(new_question)
print(new_result)
print("\nRelevant chunks from the vector store:")
relevant_docs = retriever.invoke(new_question)
for i, doc in enumerate(relevant_docs):
	print(f"\nDocument {i+1}:\n{doc.page_content}\n")
new_result

AI companions are applications that use artificial intelligence technologies to provide assistance, support, or companionship to users. They can be in the form of chatbots, virtual assistants, or social robots. AI companions interact with users in a human-like manner through natural language processing and machine learning techniques. They can perform tasks, answer questions, provide recommendations, and have conversations with users. The AI technology behind these companions allows them to continuously learn and improve their interactions with users over time.

Relevant chunks from the vector store:

Document 1:
AI companions are applications that leverage artificial intelligence technologies to provide assistance, support, or companionship to users. These AI companions can take various forms, including chatbots, virtual assistants, and social robots. They are designed to interact with users in a human-like manner
, often using natural language processing and machine learning techniqu

'AI companions are applications that use artificial intelligence technologies to provide assistance, support, or companionship to users. They can be in the form of chatbots, virtual assistants, or social robots. AI companions interact with users in a human-like manner through natural language processing and machine learning techniques. They can perform tasks, answer questions, provide recommendations, and have conversations with users. The AI technology behind these companions allows them to continuously learn and improve their interactions with users over time.'

## Conversational Memory

In [44]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage

In [45]:
# Create a prompt that includes conversation history
contextualize_q_system_prompt = """Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history. Do NOT answer the question, just reformulate it if needed and otherwise return it as is.
"""

In [46]:
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}")
])

In [47]:
# Create a history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=contextualize_q_prompt
)

In [48]:
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000258852C7620>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(ta

In [49]:
qa_system_prompt = """You are an expert assistant. Use the following context to answer the question.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    ("user", "{input}")
])

question_aware_chain = create_retrieval_chain(
    history_aware_retriever,
    create_stuff_documents_chain(llm, qa_prompt)
)

In [50]:
question_aware_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000258852C7620>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')]

In [51]:
# FIXED VERSION - Run this cell instead
# Define the prompt correctly
qa_system_prompt = """You are an expert assistant. Use the following context to answer the question.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    ("user", "{input}")
])

# Create the chain
question_aware_chain = create_retrieval_chain(
    history_aware_retriever,
    create_stuff_documents_chain(llm, qa_prompt)
)

# Test it
chat_history = []
result1 = question_aware_chain.invoke({
    "input": "What is generative AI?",  
    "chat_history": chat_history
})
print("Q1:", "What is generative AI?")
print("A1:", result1["answer"])

Q1: What is generative AI?
A1: Generative AI refers to a family of machine learning systems that learn statistical patterns from data and then produce new content—text, images, audio, code, or structured outputs—that resembles what they learned. In teaching the topic, it helps to start with the core idea that these models do not “retrieve” answers in the same way a search engine does. Instead, they estimate the most likely continuation (or completion) of an input based on the distribution they learned during training.


In [52]:
# Follow-up question that references chat history
# First, add the previous conversation to chat_history
from langchain_core.messages import HumanMessage, AIMessage

chat_history.append(HumanMessage(content="What is generative AI?"))
chat_history.append(AIMessage(content=result1["answer"]))

result2 = question_aware_chain.invoke({
    "input": "How does it work?",  
    "chat_history": chat_history
})
print("Q2:", "How does it work?")
print("A2:", result2["answer"])

Q2: How does it work?
A2: Generative AI works by first training on a dataset to learn the statistical patterns present in the data. Once trained, the model can generate new content by estimating the most likely continuation of an input based on the patterns it learned during training. This process involves sampling from the learned distribution to create new text, images, audio, code, or other outputs that resemble the data it was trained on. Overall, generative AI uses statistical patterns to generate new content rather than retrieving pre-existing answers like a search engine.


## GROQ LLM Implementation

In [58]:
os.getenv("GROQ_API_KEY")

In [59]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [63]:
# Check if GROQ_API_KEY is set
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ API key found!")
else:
    print("WARNING: GROQ_API_KEY not found in environment. Please add it to your .env file.")

GROQ API key found!


In [64]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [65]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025888BD7E00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002588B308050>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [67]:
# Alternate way
llm = init_chat_model("groq:llama-3.1-8b-instant") 
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002588B314690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002588B314B90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))